# Обучение и проверка (Colab, GPU)

Тонкий блокнот: весь код в пакете `anemia/` и в `scripts/` репозитория. Запускайте ячейки по порядку.

Архив CP-AnemiC (`CP-AnemiC_dataset.zip`, общая папка проекта `md/history/097_2026-10-02_dataset_cp_anemic/`):
положите в `MyDrive/anemia/` на Диске или перетащите в панель «Файлы» слева; иначе ячейка 2 попросит загрузить его.
Данные и модели хранятся на Диске в `MyDrive/anemia/` (без Диска — в `/content/anemia`), в репозиторий не попадают.

**Обновление кода.** Ячейка 1 при каждом запуске подтягивает `origin/main` (галочка «Обновить_код»), печатает новые коммиты и статус тестов GitHub. Правки, сделанные прямо в `/content/repo`, не теряются: они уходят в `git stash`. Код новых коммитов выполняется только когда вы запускаете ячейки 2–3.

In [ ]:
#@title Ячейка 1 — репозиторий, зависимости, хранилище
#@markdown Google Диск сохраняет данные и модели между сессиями. Если Диск не подключается — снимите галочку:
#@markdown всё будет в /content (стирается при закрытии сессии, данные скачаются заново).
Использовать_Google_Диск = False  #@param {type:"boolean"}
#@markdown **Код из GitHub.** Галочка «Обновить_код» подтягивает свежую версию ветки и печатает список новых коммитов;
#@markdown без галочки остаётся уже скачанная версия. «Коммит» — закрепить конкретный коммит (пусто = последний в ветке).
Обновить_код = True  #@param {type:"boolean"}
Ветка = 'main'  #@param {type:"string"}
Коммит = ''  #@param {type:"string"}
import os, subprocess, sys, time, json, shutil, urllib.request
REPO = 'zhansaya-afk/jjghg'
REPO_DIR = '/content/repo'
BRANCH = Ветка.strip() or 'main'
BASE = '/content/anemia'
if Использовать_Google_Диск:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE = '/content/drive/MyDrive/anemia'
    except Exception as e:
        print('⚠️ Google Диск не подключился, работаю без него (данные в /content/anemia):', str(e)[:150])
os.makedirs(BASE, exist_ok=True)
os.environ['ANEMIA_BASE'] = BASE
os.environ['ANEMIA_DATA'] = f'{BASE}/data'
os.environ['ANEMIA_MODELS'] = f'{BASE}/models'

HIST = 'results/metrics_history.csv'   # история метрик: её дописывает ячейка 3, при обновлении кода не прячем


def history_extra(head_text, local_text):
    """Строки, дописанные к истории после коммита (local = HEAD + новые строки) → текст, иначе None."""
    if head_text and head_text.endswith('\n') and local_text.startswith(head_text):
        return local_text[len(head_text):]
    return None


def git(*args):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f'git {" ".join(args)}: {r.stderr.strip()[:300]}')
    return r.stdout.strip()

# токен нужен, только если репозиторий приватный: Colab → ключ слева → Secrets → GITHUB_TOKEN
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], check=True)
    if Коммит.strip():
        git('reset', '-q', '--hard', Коммит.strip())
elif Обновить_код:
    git('fetch', '-q', 'origin', BRANCH)
    target = git('rev-parse', '--verify', (Коммит.strip() or f'origin/{BRANCH}') + '^{commit}')
    stamp = time.strftime('%Y%m%d-%H%M%S')
    # новые строки истории метрик: резервная копия в хранилище, файл — к версии HEAD, строки — обратно после обновления
    hist_rows, hist_path = None, f'{REPO_DIR}/{HIST}'
    hist_backup = f'{BASE}/metrics_history_local-{stamp}.csv'
    if os.path.isfile(hist_path) and git('status', '--porcelain', '--', HIST):   # нет файла → как раньше, без копии
        shutil.copy2(hist_path, hist_backup)
        head = subprocess.run(['git', '-C', REPO_DIR, 'show', f'HEAD:{HIST}'], capture_output=True, text=True).stdout
        hist_rows = history_extra(head, open(hist_path, encoding='utf-8').read())
        if hist_rows is None:
            print(f'⚠️ {HIST} изменён не дописыванием строк — уйдёт в stash; копия: {hist_backup}')
        else:
            git('checkout', 'HEAD', '--', HIST)   # и рабочая копия, и индекс: в stash строк не будет, дублей нет
    # ничего не теряем: правки в /content/repo уходят в stash, свои коммиты — в ветку colab-backup-*
    if git('status', '--porcelain'):
        git('stash', 'push', '-u', '-m', f'colab-{stamp}')
        print(f'⚠️ Локальные правки в {REPO_DIR} сохранены: git -C {REPO_DIR} stash list')
    if git('log', '--oneline', 'HEAD', '--not', target, f'origin/{BRANCH}'):
        git('branch', f'colab-backup-{stamp}')
        print(f'⚠️ Коммиты, которых нет в {BRANCH}, сохранены в ветке colab-backup-{stamp}')
    incoming = git('log', '--oneline', '--no-decorate', f'HEAD..{target}')
    git('reset', '-q', '--hard', target)
    if hist_rows and not os.path.isfile(hist_path):
        print(f'⚠️ В новой версии нет {HIST}: ваши строки не дописаны, они в {hist_backup}')
    elif hist_rows:
        with open(hist_path, encoding='utf-8') as f:
            new_text = f.read()
        with open(hist_backup, encoding='utf-8') as f:
            same_header = new_text.split('\n', 1)[0] == f.readline().rstrip('\n')
        if same_header:
            with open(hist_path, 'a', encoding='utf-8') as f:
                f.write(('' if new_text.endswith('\n') else '\n') + hist_rows)   # строки не склеиваются
            print(f'Строки истории метрик, дописанные в Colab, сохранены в {HIST}: {hist_rows.count(chr(10))}')
        else:
            print(f'⚠️ У {HIST} в новой версии другие колонки: ваши строки не дописаны, они в {hist_backup}')
    if Коммит.strip():
        print('Закреплён коммит', Коммит.strip())
    print('Новые коммиты:\n' + incoming if incoming else 'Новых коммитов нет.')
else:
    print('Обновление выключено: работаю на уже скачанной версии.')
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data,dev]'], check=True)
sha = git('rev-parse', 'HEAD')
print('Хранилище:', BASE)
print('Коммит:', git('log', '-1', '--format=%h %s (%cd)', '--date=short'))
# статус тестов GitHub Actions для этого коммита (без токена, только для публичного репозитория)
try:
    with urllib.request.urlopen(f'https://api.github.com/repos/{REPO}/commits/{sha}/check-runs', timeout=10) as r:
        runs = json.load(r).get('check_runs', [])
    tests = [c for c in runs if c['name'].startswith('pytest')]
    if not tests:
        print('Тесты GitHub: нет данных для этого коммита')
    elif all(c['conclusion'] == 'success' for c in tests):
        print('Тесты GitHub: ✅ пройдены')
    elif any(c['status'] != 'completed' for c in tests):
        print('Тесты GitHub: ⏳ ещё идут')
    else:
        print('Тесты GitHub: ❌ упали — сообщите Claude, прежде чем запускать дальше')
except Exception as e:
    print('Тесты GitHub: статус не получен:', str(e)[:80])
if 'anemia' in sys.modules:
    print('ℹ️ Пакет anemia уже был импортирован: чтобы взять новый код, «Среда выполнения → Перезапустить сеанс» и снова ячейка 1.')

In [ ]:
#@title Ячейка 2 — данные: скачать (один раз) и собрать manifest.csv
import os, glob, shutil
D, BASE = os.environ['ANEMIA_DATA'], os.environ['ANEMIA_BASE']
if not os.path.exists(f'{D}/eyes_defy_anemia/labels.csv'):
    !python scripts/fetch_data.py --eyes-defy
if not os.path.exists(f'{D}/cp_anemic/labels.csv'):
    # архив CP-AnemiC ищется на Диске (MyDrive/anemia/) или в /content (панель «Файлы» слева);
    # если его нигде нет — появится кнопка загрузки с компьютера
    found = sorted(glob.glob(f'{BASE}/*AnemiC*.zip') + glob.glob('/content/*AnemiC*.zip'))
    if not found:
        try:
            from google.colab import files
            print('Загрузите CP-AnemiC_dataset.zip (5 МБ):')
            up = files.upload()
            found = [os.path.abspath(n) for n in up if n.lower().endswith('.zip')]
        except Exception as e:
            print('⚠️ загрузка не удалась:', str(e)[:120])
    if found:
        cp_zip = found[0]
        !python scripts/fetch_data.py --cp-zip "{cp_zip}"
    else:
        print('⚠️ CP-AnemiC пропущен: нет архива')
!python scripts/prepare_data.py
!python -m pytest -q

In [ ]:
#@title Ячейка 3 — честный baseline (этап 1): сегментация v11 + Hb по цвету, все датасеты
# ≈10–20 мин на CPU/GPU Colab. Пришлите Claude весь вывод ячейки: таблицу и строки CSV в конце.
import os, pandas as pd
before = len(pd.read_csv('results/metrics_history.csv')) if os.path.exists('results/metrics_history.csv') else 0
!python scripts/evaluate.py --tag baseline_v11
h = pd.read_csv('results/metrics_history.csv')
new = h.iloc[before:]
new.to_csv(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv", mode='a', index=False,
           header=not os.path.exists(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv"))
print('\n--- строки для results/metrics_history.csv ---')
print(new.to_csv(index=False))

## Ядро HemaVision: фото → риск анемии

Нужны ячейки 1–2 (код и `manifest.csv`). **Ячейка Я1** один раз: честная проверка на фолдах по пациентам
(взрослые и дети раздельно) и обучение итоговой модели → `models/hemavision_core.joblib`.
**Ячейка Я2** — сколько угодно раз: одно фото → риск. **Ячейка Я3** — готовый анализ крови, отдельно от фото.

In [ ]:
#@title Ячейка Я1 — проверка ядра (CV по пациентам) и обучение итоговой модели
# Признаки — общий кэш с ячейкой 3; первый запуск после обновления кода считает их заново (≈8 мин на Eyes-defy).
import os
!python scripts/evaluate.py --task core --tag core
!python scripts/train_core.py
print('Модель:', os.path.join(os.environ['ANEMIA_MODELS'], 'hemavision_core.joblib'))

In [ ]:
#@title Ячейка Я2 — одно фото → оценка риска { display-mode: "form" }
#@markdown Фото: глаз с оттянутым нижним веком крупным планом (или уже вырезанная конъюнктива).
#@markdown Путь пустой — откроется окно загрузки файла.
Путь_к_фото = ''  #@param {type:"string"}
Пол = 'Женский'  #@param ["Женский", "Мужской", "Не указан"]
Возраст = 30  #@param {type:"number"}
Беременность = False  #@param {type:"boolean"}
import matplotlib.pyplot as plt
from anemia.predict import analyze_photo, format_result, overlay
src = Путь_к_фото.strip()
if not src:
    from google.colab import files
    up = files.upload()
    src = next(iter(up.values()))          # байты файла: analyze_photo принимает путь, байты, PIL или массив
r = analyze_photo(src, sex={'Женский': 'F', 'Мужской': 'M'}.get(Пол), age=Возраст, pregnant=Беременность)
vis = overlay(r)
if vis is not None:
    plt.figure(figsize=(5, 4)); plt.imshow(vis); plt.axis('off')
    plt.title('зелёный — конъюнктива, голубой — склера', fontsize=9); plt.show()
print(format_result(r))

In [ ]:
#@title Ячейка Я3 — готовый анализ крови (отдельно от фото) { display-mode: "form" }
Hb = '12.5'  #@param {type:"string"}
Единица = 'г/дл'  #@param ["г/л", "г/дл", "ммоль/л"]
Пол_анализ = 'Женский'  #@param ["Женский", "Мужской", "Не указан"]
Возраст_анализ = 30  #@param {type:"number"}
Беременность_анализ = False  #@param {type:"boolean"}
from anemia.labs import lab_hb_report
lab = lab_hb_report(Hb, Единица, sex={'Женский': 'F', 'Мужской': 'M'}.get(Пол_анализ), age=Возраст_анализ,
                    pregnant=Беременность_анализ)
print(lab['text'])